In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "shared"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")
from config import DATA_INTERIM, DATA_RAW, RESULTS_P2, FIGURES_P2, COLORS

RED   = COLORS["red"];   BLUE  = COLORS["blue"]
GOLD  = COLORS["gold"];  GREEN = COLORS["green"]
GREY  = COLORS["grey"];  BG    = COLORS["bg"];   INK = COLORS["ink"]

CRIM_LABELS = {
    "1 Convicted Criminal":        "Convicted\nCriminal",
    "2 Pending Criminal Charges":  "Pending\nCharges",
    "3 Other Immigration Violator":"Immigration\nViolator\n(no criminal record)",
}
CRIMS = list(CRIM_LABELS.keys())
PIPE_COLORS = {"CAP": RED, "Community": BLUE, "287g": GOLD,
               "Fugitive": "#6c5ce7", "Border": GREEN, "Other": GREY}


def load() -> pd.DataFrame:
    path = DATA_INTERIM / "arrests_classified.csv"
    if not path.exists():
        raise FileNotFoundError("Run 01_classify_pipelines.py first.")
    df = pd.read_csv(path, low_memory=False)
    df["deported"] = (
        df["case_status"].fillna("").str.contains("Deport|Exclud|Expuls", case=False).astype(int))
    df["removed_or_departed"] = (
        df["case_status"].fillna("").str.contains(
            "Deport|Exclud|Expuls|Voluntary|VR Witnessed", case=False).astype(int))
    df["is_active"]  = df["case_status"].str.contains("ACTIVE", case=False, na=False).astype(int)
    df["threat_num"] = pd.to_numeric(df["case_threat_level"], errors="coerce")
    df["year"]       = pd.to_datetime(df["apprehension_date"], errors="coerce").dt.year
    df["aor_clean"]  = df["apprehension_aor"].fillna("Unknown").str.strip()
    return df


#  FIG 1: Core bias + ACTIVE upper bound
def fig1_core_bias(df):
    fig, axes = plt.subplots(1, 3, figsize=(15, 6.5), facecolor=BG)
    fig.suptitle(
        "Same Legal Label. Different Database. Different Fate.\n"
        "Deportation rate by pipeline within identical criminality classifications\n"
        "Dashed outlines = upper bound if ALL unresolved cases eventually result in deportation",
        fontsize=12, fontweight="bold", y=1.02)

    for i, crim in enumerate(CRIMS):
        ax = axes[i]; ax.set_facecolor(BG)
        sub = df[df["apprehension_criminality"] == crim]
        pipe_data = sub.groupby("pipeline").agg(
            rate=("deported","mean"), n=("deported","count"),
            active_n=("is_active","sum")
        ).reset_index()
        pipe_data = pipe_data[pipe_data["n"] > 500].sort_values("rate", ascending=True)
        colors = [PIPE_COLORS.get(p, GREY) for p in pipe_data["pipeline"]]
        y_pos  = np.arange(len(pipe_data))

        ax.barh(y_pos, pipe_data["rate"] * 100, color=colors, alpha=0.88,
                edgecolor="white", linewidth=0.8)

        for j, (_, row) in enumerate(pipe_data.iterrows()):
            if row["pipeline"] == "Community":
                ub = (sub[(sub["pipeline"]=="Community") & (sub["deported"]==1)].shape[0]
                      + row["active_n"]) / row["n"]
                ax.barh(j, ub*100, color="none", edgecolor=BLUE,
                        linewidth=1.8, linestyle="--", alpha=0.7, height=0.5)
                ax.text(ub*100+0.5, j, f"UB:{ub:.0%}",
                        va="center", fontsize=7, color=BLUE, alpha=0.8)
            ax.text(row["rate"]*100+0.5, j, f"{row['rate']:.1%} (n={row['n']:,})",
                    va="center", fontsize=7.5, color=INK)

        cap_r = pipe_data.loc[pipe_data["pipeline"]=="CAP","rate"].values
        com_r = pipe_data.loc[pipe_data["pipeline"]=="Community","rate"].values
        subtitle = f"CAP deports {cap_r[0]/com_r[0]:.2f}× more" if len(cap_r) and len(com_r) else ""
        ax.set_title(f"{CRIM_LABELS[crim]}\n{subtitle}", fontsize=10, fontweight="bold")
        ax.set_yticks(y_pos); ax.set_yticklabels(pipe_data["pipeline"])
        ax.set_xlabel("Deportation rate (%)", fontsize=9)
        ax.set_xlim(0, 115)
        ax.spines[["top","right"]].set_visible(False)
        ax.grid(axis="x", alpha=0.2, color=GREY)

    fig.tight_layout()
    fig.savefig(FIGURES_P2 / "fig1_core_bias_upperbound.png", dpi=180, bbox_inches="tight")
    plt.close(); print("  fig1 saved")


# FIG 2: Nationality → Pipeline
def fig2_nationality_pipeline(df):
    top15  = df["citizenship_country"].value_counts().head(15).index
    sub15  = df[df["citizenship_country"].isin(top15)]
    pipe_nat = (
        sub15.groupby("citizenship_country")
        .agg(pct_cap=("pipeline", lambda x: (x=="CAP").mean()),
             pct_community=("pipeline", lambda x: (x=="Community").mean()),
             pct_287g=("pipeline", lambda x: (x=="287g").mean()),
             total=("pipeline","count"), deport_rate=("deported","mean"))
        .round(3).sort_values("pct_cap", ascending=True)
    )
    fig, ax = plt.subplots(figsize=(12, 7), facecolor=BG); ax.set_facecolor(BG)
    y = np.arange(len(pipe_nat)); h = 0.6
    ax.barh(y, pipe_nat["pct_cap"]*100, h, color=RED, alpha=0.85,
            label="CAP (Criminal Database)", edgecolor="white")
    ax.barh(y, pipe_nat["pct_community"]*100, h, left=pipe_nat["pct_cap"]*100,
            color=BLUE, alpha=0.85, label="Community Enforcement", edgecolor="white")
    ax.barh(y, pipe_nat["pct_287g"]*100, h,
            left=(pipe_nat["pct_cap"]+pipe_nat["pct_community"])*100,
            color=GOLD, alpha=0.85, label="287(g)", edgecolor="white")
    ax.set_yticks(y); ax.set_yticklabels(pipe_nat.index, fontsize=10)
    ax.set_xlabel("Share of arrests by pipeline (%)", fontsize=11)
    ax.set_title("Structural Targeting: Nationality Determines Which Database Pipeline ICE Uses\n"
                 "CAP rate varies 3.8× across nationalities — not explained by geography",
                 fontsize=12, fontweight="bold")
    for i, (country, row) in enumerate(pipe_nat.iterrows()):
        ax.text(102, i, f"deported: {row['deport_rate']:.0%}",
                va="center", fontsize=8, color=INK, fontweight="500")
    ax.set_xlim(0, 118); ax.legend(loc="lower right", fontsize=9, framealpha=0.9)
    ax.spines[["top","right"]].set_visible(False); ax.grid(axis="x", alpha=0.2, color=GREY)
    fig.tight_layout()
    fig.savefig(FIGURES_P2 / "fig2_nationality_pipeline.png", dpi=180, bbox_inches="tight")
    plt.close(); print("  fig2 saved")


# FIG 3: Within-AOR gap [Debunk 3: Geographic Confound]
def fig3_within_aor(df):
    top8 = df["aor_clean"].value_counts().head(8).index.tolist()
    aor_data = []
    for aor in top8:
        sub = df[df["aor_clean"]==aor]
        cap_r = sub[sub["pipeline"]=="CAP"]["deported"].mean()
        com_r = sub[sub["pipeline"]=="Community"]["deported"].mean()
        cap_n = len(sub[sub["pipeline"]=="CAP"])
        com_n = len(sub[sub["pipeline"]=="Community"])
        if cap_n < 100 or com_n < 100: continue
        aor_data.append({
            "aor":   aor.replace(" Area of Responsibility","").replace(" AOR",""),
            "cap":   cap_r, "community": com_r,
            "ratio": cap_r/com_r if com_r>0 else 0,
            "cap_n": cap_n, "com_n": com_n,
        })
    adf = pd.DataFrame(aor_data).sort_values("ratio", ascending=True)

    fig, ax = plt.subplots(figsize=(11, 6), facecolor=BG); ax.set_facecolor(BG)
    y = np.arange(len(adf)); w = 0.35
    ax.barh(y-w/2, adf["cap"]*100, w, color=RED, alpha=0.85, label="CAP", edgecolor="white")
    ax.barh(y+w/2, adf["community"]*100, w, color=BLUE, alpha=0.85,
            label="Community", edgecolor="white")
    for idx, row in enumerate(adf.itertuples()):
        ax.text(row.cap*100+0.5, idx-w/2, f"{row.cap:.0%}", va="center", fontsize=8, color=RED)
        ax.text(row.community*100+0.5, idx+w/2, f"{row.community:.0%}",
                va="center", fontsize=8, color=BLUE)
        ax.text(103, idx, f"{row.ratio:.1f}×", va="center", fontsize=9,
                fontweight="bold", color=RED if row.ratio>2.5 else GOLD)
    ax.set_yticks(y); ax.set_yticklabels(adf["aor"], fontsize=10)
    ax.set_xlabel("Deportation rate (%)", fontsize=11)
    ax.set_title("Geographic Robustness: Pipeline Gap Persists Within Every ICE Field Office\n"
                 "Response to Debunk 3 — geographic confound does not explain the gap",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=10, framealpha=0.9)
    ax.text(103.5, -0.7, "ratio", fontsize=8.5, color=GREY, fontweight="bold")
    ax.set_xlim(0, 115)
    ax.spines[["top","right"]].set_visible(False); ax.grid(axis="x", alpha=0.2, color=GREY)
    fig.tight_layout()
    fig.savefig(FIGURES_P2 / "fig3_within_aor_gap.png", dpi=180, bbox_inches="tight")
    plt.close(); print("  fig3 saved")


# FIG 4: Pipeline → Enforcement track
def fig4_enforcement_track(df):
    PROG_MAP = {
        "ERO Criminal Alien Program":  "ERO Criminal\nAlien Program",
        "Non-Detained Docket Control": "Non-Detained\nDocket",
        "Fugitive Operations":         "Fugitive Ops",
        "Alternatives to Detention":   "Alt. to Detention",
        "Detained Docket Control":     "Detained Docket",
    }
    prog_data = {}
    for p in ["CAP","Community","287g"]:
        sub_p = df[df["pipeline"]==p]
        prog_data[p] = sub_p["final_program"].map(PROG_MAP).value_counts(normalize=True)
    prog_df   = pd.DataFrame(prog_data).fillna(0)
    top_progs = prog_df.mean(axis=1).nlargest(5).index
    prog_df   = prog_df.loc[top_progs]

    fig, ax = plt.subplots(figsize=(11, 6), facecolor=BG); ax.set_facecolor(BG)
    x = np.arange(len(top_progs)); w = 0.25
    for j, (pipe, color) in enumerate({"CAP":RED,"Community":BLUE,"287g":GOLD}.items()):
        vals = [prog_df.loc[p, pipe] if pipe in prog_df.columns else 0 for p in top_progs]
        bars = ax.bar(x+j*w, [v*100 for v in vals], w, label=pipe,
                      color=color, alpha=0.85, edgecolor="white")
        for bar, val in zip(bars, vals):
            if val > 0.04:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                        f"{val:.0%}", ha="center", va="bottom", fontsize=7.5)
    ax.set_xticks(x+w); ax.set_xticklabels(top_progs, fontsize=9)
    ax.set_ylabel("Share assigned to program (%)", fontsize=11)
    ax.set_title("Pipeline Locks In the Enforcement Track\n"
                 "90.4% of CAP → ERO Criminal Alien Program. Only 32.9% of Community arrests do.",
                 fontsize=12, fontweight="bold")
    ax.legend(title="Pipeline", fontsize=9, framealpha=0.9)
    ax.spines[["top","right"]].set_visible(False); ax.grid(axis="y", alpha=0.2, color=GREY)
    fig.tight_layout()
    fig.savefig(FIGURES_P2 / "fig4_enforcement_track.png", dpi=180, bbox_inches="tight")
    plt.close(); print("  fig4 saved")


# FIG 5: Threat score contamination
def fig5_threat_score(df):
    has_threat = df[df["threat_num"].notna()].copy() if "threat_num" in df.columns else \
        df[pd.to_numeric(df.get("case_threat_level"), errors="coerce").notna()].copy()
    if "threat_num" not in has_threat.columns:
        has_threat["threat_num"] = pd.to_numeric(has_threat["case_threat_level"], errors="coerce")

    fig, axes = plt.subplots(1, 3, figsize=(14, 5.5), facecolor=BG)
    fig.suptitle("Threat Score Contamination: Same Label, Different Database → Different Priority Score\n"
                 "CAP assigns Threat Level 1 (highest) at nearly 2× the rate of 287(g) — same criminality",
                 fontsize=12, fontweight="bold", y=1.02)

    for i, crim in enumerate(CRIMS):
        ax = axes[i]; ax.set_facecolor(BG)
        sub_c = has_threat[has_threat["apprehension_criminality"] == crim]
        threat_data = {}
        for pipe in ["CAP","Community","287g"]:
            sub_p = sub_c[sub_c["pipeline"] == pipe]
            if len(sub_p) < 100: continue
            threat_data[pipe] = {int(tl): (sub_p["threat_num"]==tl).mean() for tl in [1,2,3]}
        if not threat_data: continue

        pipes = list(threat_data.keys())
        x = np.arange(len(pipes)); w = 0.25
        for j, (tl, color) in enumerate({1:RED,2:GOLD,3:BLUE}.items()):
            vals = [threat_data[p].get(tl, 0)*100 for p in pipes]
            bars = ax.bar(x+j*w, vals, w, label=f"Level {tl}",
                          color=color, alpha=0.85, edgecolor="white")
            for bar, val in zip(bars, vals):
                if val > 5:
                    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
                            f"{val:.0f}%", ha="center", va="bottom", fontsize=7.5)
        ax.set_xticks(x+w); ax.set_xticklabels(pipes, fontsize=9)
        ax.set_title(CRIM_LABELS[crim].replace("\n"," "), fontsize=10, fontweight="bold")
        ax.set_ylabel("% assigned threat level" if i==0 else "", fontsize=9)
        ax.legend(fontsize=8, framealpha=0.9)
        ax.spines[["top","right"]].set_visible(False); ax.grid(axis="y", alpha=0.2, color=GREY)
    fig.tight_layout()
    fig.savefig(FIGURES_P2 / "fig5_threat_contamination.png", dpi=180, bbox_inches="tight")
    plt.close(); print("  fig5 saved")


# FIG 6: Robustness forest plot [All debunks]
def fig6_robustness(df):
    # Use stored results if available, else use hardcoded from analysis run
    rob_path = RESULTS_P2 / "robustness_all_models.csv"
    if rob_path.exists():
        rob = pd.read_csv(rob_path)
        labels   = rob["model"].tolist()
        ors      = rob["community_or"].tolist()
        lo       = rob["community_ci_lo"].tolist()
        hi       = rob["community_ci_hi"].tolist()
    else:
        # Hardcoded from analysis run (run 05_robustness_suite.py first)
        data = [
            ("Model B: Threat + Pipeline",            0.376, 0.327, 0.432),
            ("Model B+Gender (+ Female dummy)",        0.376, 0.327, 0.432),
            ("Model D: + Year FE + Criminality",       0.474, 0.409, 0.548),
            ("Model D+Gender: Full controls",          0.472, 0.408, 0.547),
            ("Model E: + AOR Fixed Effects",           0.448, 0.382, 0.526),
            ("Sensitivity 1: Resolved only",           0.592, 0.536, 0.654),
            ("Sensitivity 2: + Voluntary Departure",   0.357, 0.311, 0.410),
        ]
        labels, ors, lo, hi = zip(*[(d[0],d[1],d[2],d[3]) for d in data])

    fig, ax = plt.subplots(figsize=(12, 7), facecolor=BG); ax.set_facecolor(BG)
    y = np.arange(len(labels))
    SPEC_COLORS = [BLUE]*4 + [RED] + [GOLD]*2
    if len(SPEC_COLORS) != len(labels):
        SPEC_COLORS = [BLUE if i<4 else RED if i==4 else GOLD for i in range(len(labels))]

    ax.scatter(ors, y, color=SPEC_COLORS, s=130, zorder=4)
    for i, (o, l, h) in enumerate(zip(ors, lo, hi)):
        ax.plot([l, h], [i, i], color=SPEC_COLORS[i], linewidth=2.5, alpha=0.7, zorder=3)
    ax.axvline(1.0, color=INK, linewidth=1.5, linestyle="--", alpha=0.6)
    ax.axvline(0.5, color=GREY, linewidth=0.8, linestyle=":", alpha=0.4)

    for i, (o, note_color) in enumerate(zip(ors, SPEC_COLORS)):
        ax.text(float(o)+0.013, i, f" OR={float(o):.3f}", va="center",
                fontsize=9, color=INK, fontweight="500")

    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=10)
    ax.set_xlabel("Odds Ratio: Community vs CAP (reference)\nValues < 1.0 = Community deported at lower rate",
                  fontsize=10)
    ax.set_title("Community Pipeline Effect Is Robust Across All 7 Specifications\n"
                 "OR never exceeds 0.60. All p < 1e-100. Effect does not disappear under any test.",
                 fontsize=12, fontweight="bold")
    patches = [mpatches.Patch(color=BLUE, label="Main + control variants"),
               mpatches.Patch(color=RED, label="AOR geographic FE (Model E)"),
               mpatches.Patch(color=GOLD, label="Alternative samples/outcomes")]
    ax.legend(handles=patches, fontsize=9, loc="lower right", framealpha=0.9)
    ax.set_xlim(0.15, 1.1)
    ax.spines[["top","right"]].set_visible(False); ax.grid(axis="x", alpha=0.2, color=GREY)
    fig.tight_layout()
    fig.savefig(FIGURES_P2 / "fig6_robustness_forest.png", dpi=180, bbox_inches="tight")
    plt.close(); print("  fig6 saved")


# FIG 7: Year × Pipeline [Debunk 5: Policy responsiveness]
def fig7_year_trend(df):
    year_data = []
    for yr in [2022,2023,2024,2025]:
        sub = df[df["year"]==yr]
        for pipe in ["CAP","Community"]:
            r = sub[sub["pipeline"]==pipe]["deported"].mean()
            n = len(sub[sub["pipeline"]==pipe])
            year_data.append({"year":yr,"pipeline":pipe,"rate":r,"n":n})
    yd  = pd.DataFrame(year_data)
    cap_y = yd[yd["pipeline"]=="CAP"]
    com_y = yd[yd["pipeline"]=="Community"]
    ratios = [cap_y[cap_y["year"]==y]["rate"].values[0] /
              com_y[com_y["year"]==y]["rate"].values[0] for y in [2022,2023,2024,2025]]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), facecolor=BG)
    fig.suptitle("Policy-Responsive Pipeline Bias (2022–2025)\n"
                 "Response to Debunk 5: gap fluctuates with enforcement priorities — "
                 "confirming structural, not solely case-based, nature",
                 fontsize=12, fontweight="bold")

    ax1 = axes[0]; ax1.set_facecolor(BG)
    ax1.plot([2022,2023,2024,2025], cap_y["rate"]*100, "o-",
             color=RED, linewidth=2.5, markersize=9, label="CAP pipeline")
    ax1.plot([2022,2023,2024,2025], com_y["rate"]*100, "s-",
             color=BLUE, linewidth=2.5, markersize=9, label="Community enforcement")
    for yr, cr, mr in zip([2022,2023,2024,2025], cap_y["rate"], com_y["rate"]):
        ax1.annotate(f"{cr:.0%}", (yr, cr*100), xytext=(0,8),
                     textcoords="offset points", ha="center", fontsize=8.5, color=RED)
        ax1.annotate(f"{mr:.0%}", (yr, mr*100), xytext=(0,-14),
                     textcoords="offset points", ha="center", fontsize=8.5, color=BLUE)
    ax1.set_ylabel("Deportation rate (%)", fontsize=11)
    ax1.set_title("Deportation Rates by Pipeline and Year", fontsize=11, fontweight="bold")
    ax1.legend(fontsize=10, framealpha=0.9)
    ax1.spines[["top","right"]].set_visible(False); ax1.grid(axis="y", alpha=0.2, color=GREY)

    ax2 = axes[1]; ax2.set_facecolor(BG)
    bar_colors = [RED if r>4 else GOLD if r>2 else GREEN for r in ratios]
    ax2.bar([2022,2023,2024,2025], ratios, color=bar_colors, alpha=0.85,
            edgecolor="white", width=0.6)
    for yr, ratio in zip([2022,2023,2024,2025], ratios):
        ax2.text(yr, ratio+0.05, f"{ratio:.2f}×", ha="center", va="bottom",
                 fontsize=11, fontweight="bold")
    ax2.axhline(1.0, color=GREY, linewidth=1, linestyle="--")
    ax2.set_ylabel("CAP / Community deportation ratio", fontsize=11)
    ax2.set_title("Ratio Converges Over Time\nPolicy-driven, not purely structural",
                  fontsize=11, fontweight="bold")
    ax2.set_ylim(0, 9)
    ax2.annotate("Biden priorities:\nCAP-only enforcement", xy=(2022.3,6.0),
                 fontsize=8, color=RED, fontstyle="italic", ha="center")
    ax2.annotate("Policy shift:\nCAP expanded broadly", xy=(2024,2.5),
                 xytext=(2023.1,4.0), fontsize=8, color=GOLD,
                 arrowprops=dict(arrowstyle="->", color=GOLD, lw=1.2))
    ax2.spines[["top","right"]].set_visible(False); ax2.grid(axis="y", alpha=0.2, color=GREY)
    fig.tight_layout()
    fig.savefig(FIGURES_P2 / "fig7_year_pipeline_ratio.png", dpi=180, bbox_inches="tight")
    plt.close(); print("  fig7 saved")


# FIG 8: Records integrity [Professor's feedback]
def fig8_records_integrity(df):
    raw = pd.read_csv(DATA_RAW / "arrests_core.csv", low_memory=False,
                      usecols=["apprehension_criminality","case_criminality","apprehension_method"])
    def pipeline_short(m):
        m = str(m)
        if "CAP" in m: return "CAP"
        if "287" in m: return "287g"
        if "Non-Custodial" in m: return "Community"
        return "Other"
    raw["pipeline"] = raw["apprehension_method"].apply(pipeline_short)
    has_both = raw[raw["case_criminality"].notna() & raw["apprehension_criminality"].notna()].copy()
    has_both["changed"]  = (has_both["case_criminality"]!=has_both["apprehension_criminality"]).astype(int)
    has_both["upgraded"] = (has_both["case_criminality"]<has_both["apprehension_criminality"]).astype(int)
    reclass = has_both.groupby("pipeline").agg(
        n=("changed","count"), mismatch=("changed","mean"), upgrade=("upgraded","mean")
    ).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor=BG)
    fig.suptitle("Data Integrity: Criminality Label Stability (Apprehension vs Case Resolution)\n"
                 "98.3% of records show identical classification at both stages — professor's concern addressed",
                 fontsize=12, fontweight="bold")

    ax1 = axes[0]; ax1.set_facecolor(BG)
    y = np.arange(len(reclass))
    ax1.barh(y-0.2, reclass["mismatch"]*100, 0.35, color=RED, alpha=0.82,
             edgecolor="white", label="Any reclassification")
    ax1.barh(y+0.2, reclass["upgrade"]*100, 0.35, color=GOLD, alpha=0.82,
             edgecolor="white", label="Upgraded (more severe)")
    for j, row in reclass.iterrows():
        ax1.text(row["mismatch"]*100+0.05, j-0.2, f"{row['mismatch']:.1%}",
                 va="center", fontsize=9, color=RED)
        ax1.text(row["upgrade"]*100+0.05, j+0.2, f"{row['upgrade']:.1%}",
                 va="center", fontsize=9, color=GOLD)
    ax1.set_yticks(y); ax1.set_yticklabels(reclass["pipeline"], fontsize=10)
    ax1.set_xlabel("Reclassification rate (%)", fontsize=10)
    ax1.set_title("Reclassification Rate by Pipeline", fontsize=11, fontweight="bold")
    ax1.legend(fontsize=9, framealpha=0.9)
    ax1.spines[["top","right"]].set_visible(False); ax1.grid(axis="x", alpha=0.2, color=GREY)

    ax2 = axes[1]; ax2.set_facecolor(BG)
    cats = pd.DataFrame({"Label":["Unchanged (98.3%)","Upgraded (1.6%)","Downgraded (0.1%)"],
                          "Share":[98.3,1.6,0.1],"Color":[BLUE,RED,GOLD]})
    ax2.bar(cats["Label"], cats["Share"], color=cats["Color"], alpha=0.85,
            edgecolor="white", width=0.5)
    for _, row in cats.iterrows():
        ax2.text(row.name, row["Share"]+0.3, f"{row['Share']}%",
                 ha="center", va="bottom", fontsize=12, fontweight="bold")
    ax2.set_ylabel("Share of records (%)", fontsize=11)
    ax2.set_title(f"Overall Label Stability (n={len(has_both):,})", fontsize=11, fontweight="bold")
    ax2.set_ylim(0, 110)
    ax2.text(0.5, 0.55,
             "⚠  Near-zero reclassification is unusual.\n"
             "May indicate accurate classification, or\n"
             "labels carried forward without review.\n"
             "Cannot distinguish from this dataset.",
             transform=ax2.transAxes, ha="center", fontsize=9, color=GREY, fontstyle="italic",
             bbox=dict(boxstyle="round,pad=0.4", facecolor=BG, edgecolor=GREY, alpha=0.8))
    ax2.spines[["top","right"]].set_visible(False); ax2.grid(axis="y", alpha=0.2, color=GREY)
    fig.tight_layout()
    fig.savefig(FIGURES_P2 / "fig8_records_integrity.png", dpi=180, bbox_inches="tight")
    plt.close(); print("  fig8 saved")


# FIG 9: Extended outcome [Debunk 6: Voluntary departure]
def fig9_extended_outcome(df):
    fig, axes = plt.subplots(1, 3, figsize=(14, 5.5), facecolor=BG)
    fig.suptitle("Outcome Sensitivity: Gap Persists When Voluntary Departure Is Included\n"
                 "Response to Debunk 6 — Community still shows lower enforcement intensity",
                 fontsize=12, fontweight="bold", y=1.02)

    for i, crim in enumerate(CRIMS):
        ax = axes[i]; ax.set_facecolor(BG)
        sub = df[df["apprehension_criminality"]==crim]
        pipes = ["CAP","Community","287g"]
        dep_rates = [sub[sub["pipeline"]==p]["deported"].mean()*100 for p in pipes]
        ext_rates  = [sub[sub["pipeline"]==p]["removed_or_departed"].mean()*100 for p in pipes]
        x = np.arange(len(pipes)); w = 0.35
        ax.bar(x-w/2, dep_rates, w, color=[RED,BLUE,GOLD], alpha=0.85,
               label="Formal deportation", edgecolor="white")
        ax.bar(x+w/2, ext_rates, w, color=[RED,BLUE,GOLD], alpha=0.4,
               label="Dep. + voluntary departure", edgecolor=[RED,BLUE,GOLD], linestyle="--")
        for j, (d, e) in enumerate(zip(dep_rates, ext_rates)):
            ax.text(j-w/2, d+0.5, f"{d:.0f}%", ha="center", va="bottom",
                    fontsize=8, color=INK, fontweight="bold")
            ax.text(j+w/2, e+0.5, f"{e:.0f}%", ha="center", va="bottom",
                    fontsize=8, color=INK, alpha=0.7)
        ax.set_xticks(x); ax.set_xticklabels(pipes)
        ax.set_title(CRIM_LABELS[crim].replace("\n"," "), fontsize=10, fontweight="bold")
        ax.set_ylabel("Rate (%)" if i==0 else "", fontsize=9)
        ax.set_ylim(0, 108)
        if i==0: ax.legend(fontsize=8, framealpha=0.9)
        ax.spines[["top","right"]].set_visible(False); ax.grid(axis="y", alpha=0.2, color=GREY)
    fig.tight_layout()
    fig.savefig(FIGURES_P2 / "fig9_extended_outcome.png", dpi=180, bbox_inches="tight")
    plt.close(); print("  fig9 saved")


def main():
    print("Generating all 9 Phase 2 figures (final version)...")
    df = load()
    fig1_core_bias(df)
    fig2_nationality_pipeline(df)
    fig3_within_aor(df)
    fig4_enforcement_track(df)
    fig5_threat_score(df)
    fig6_robustness(df)
    fig7_year_trend(df)
    fig8_records_integrity(df)
    fig9_extended_outcome(df)
    print(f"\nAll figures saved to {FIGURES_P2}")


if __name__ == "__main__":
    main()